<a href="https://colab.research.google.com/github/moyx782/zh-classifier-pipeline/blob/master/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Source-Aware 中文词句分类器 — Colab 训练 notebook

本 notebook 在 Google Colab (GPU) 上完成全流程：
1. 环境准备
2. 上传本地项目 / 克隆
3. 下载原始词典数据
4. 构建 source-aware JSONL 数据集（弱监督）
5. 训练 Tfidf + LogisticRegression baseline
6. 训练 BERT-base-chinese 多标签分类器
7. 预测示例 + 对比

**运行前**：`Runtime → Change runtime type → GPU`

## 0. 环境与依赖

In [1]:
!pip -q install scikit-learn joblib numpy
!pip -q install torch transformers

In [2]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## 1. 获取项目代码

**方式 A（推荐）**：直接从 GitHub 克隆。

In [3]:
# 方式 A: git clone (推荐 — 代码总是最新)
import os
if not os.path.exists('zh-classifier-pipeline'):
    !git clone https://github.com/moyx782/zh-classifier-pipeline.git
os.chdir('/content/zh-classifier-pipeline')
print('当前目录:', os.getcwd())
print('目录内容:', os.listdir('.'))

Cloning into 'zh-classifier-pipeline'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 32 (delta 6), reused 32 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 50.03 KiB | 868.00 KiB/s, done.
Resolving deltas: 100% (6/6), done.
当前目录: /content/zh-classifier-pipeline
目录内容: ['run_all.sh', 'src', 'requirements.txt', 'train_colab.ipynb', '.gitignore', 'outputs', '.git', 'run_all.ps1', 'README.md']


In [4]:
# 方式 B (备选): 上传 project.zip 手动解压
# 如果你修改了代码想测试，先在本地打包上传 project.zip
# import zipfile
# if os.path.exists('/content/project.zip'):
#     with zipfile.ZipFile('/content/project.zip','r') as z:
#         z.extractall('/content/zh-classifier-pipeline')
#     print('project.zip 已解压覆盖')
# os.chdir('/content/zh-classifier-pipeline')
# print('目录内容:', os.listdir('.'))

In [9]:
import os, zipfile

if os.path.exists('project.zip'):
    with zipfile.ZipFile('project.zip', 'r') as z:
        z.extractall('.')
    print('project.zip 已解压')
else:
    print('没有 project.zip，请把 src/ outputs/ 等目录手动上传到 /content/')

os.chdir('/content')
print('当前目录:', os.getcwd())
print('目录内容:', os.listdir('.'))

没有 project.zip，请把 src/ outputs/ 等目录手动上传到 /content/
当前目录: /content
目录内容: ['.config', 'zh-classifier-pipeline', 'run_all.sh', 'src', 'requirements.txt', 'outputs', 'run_all.ps1', 'sample_data']


## 2. 下载原始词典数据（GitHub 主源）

约 60MB，几秒完成。

In [10]:
!python src/fetch_sources.py

fetch_sources.py — Fetch Report

[pwxcoo/chinese-xinhua]  role=primary_bootstrap_source  status=partial_or_failed
  ci: downloaded=True  records=264434  size=26129191B
  idiom: downloaded=True  records=30895  size=10319857B
  word: downloaded=True  records=16142  size=27354320B
  xiehouyu: downloaded=True  records=14032  size=1272390B

[g0v/moedict-data]  role=supplemental_source  status=partial_or_failed
  dict-revised: downloaded=True  records=163087  size=75399170B

[thunlp/THUOCL]  role=domain_enrichment_source  status=partial_or_failed
  IT: downloaded=True  records=0  size=308187B
    ERROR: JSON parse error: Expecting value: line 1 column 1 (char 0)
  caijing: downloaded=True  records=0  size=62296B
    ERROR: JSON parse error: Expecting value: line 1 column 1 (char 0)
  car: downloaded=True  records=0  size=32492B
    ERROR: JSON parse error: Expecting value: line 1 column 1 (char 0)
  chengyu: downloaded=True  records=0  size=166549B
    ERROR: JSON parse error: Expecting valu

## 3. 构建 source-aware JSONL 数据集

35 万条样本，弱监督自动打标，1-2 分钟。

In [12]:
!python src/build_dataset.py

ci.json: 264434 records -> 264434 samples
idiom.json: 30895 records -> 61788 samples
word.json: 16142 records -> 16142 samples
xiehouyu.json: 14032 records -> 42096 samples
INFO: /content/raw/g0v__moedict-data/dict-revised.json not found (optional)
build_dataset.py — Dataset build complete
Total samples: 359051
Needs review: 343203
Confidence: min=0.45, max=0.95, avg=0.5232

Source / Dataset distribution:
  pwxcoo/chinese-xinhua/ci: 264384
  pwxcoo/chinese-xinhua/idiom: 42519
  pwxcoo/chinese-xinhua/word: 14975
  pwxcoo/chinese-xinhua/xiehouyu: 37173

Cognitive label distribution:
  LANGUAGE_EXPRESSION: 359051
  DATA_INFO: 278496
  RELATION_STRUCTURE: 71862
  CONCEPT_ABSTRACT: 32376
  ENTITY_OBJECT: 25649
  ATTRIBUTE_PROPERTY: 15965
  ACTION_OPERATION: 9557
  GOAL_INTENT: 4586
  ERROR_PROBLEM: 2762
  METHOD_STRATEGY: 2168
  STATE_STATUS: 1554
  TOOL_SYSTEM: 1457

Domain label distribution:
  CHINESE_DICTIONARY: 359051
  SOFTWARE: 627
  ML: 424
  NETWORK: 406
  ROBOTICS: 64
  AGENT_TOOL

In [13]:
# 查看构建报告
import json
with open('outputs/build_report.json', encoding='utf-8') as f:
    rpt = json.load(f)
print('总样本:', rpt['total_samples'])
print('needs_review:', rpt['needs_review_count'])
print('confidence:', rpt['confidence_stats'])
print('\ncognitive 分布:')
for k, v in sorted(rpt['cognitive_label_counts'].items(), key=lambda x: -x[1]):
    print(f'  {k}: {v}')

总样本: 359051
needs_review: 343203
confidence: {'min': 0.45, 'max': 0.95, 'avg': 0.5232, 'sum': 0.0}

cognitive 分布:
  LANGUAGE_EXPRESSION: 359051
  DATA_INFO: 278496
  RELATION_STRUCTURE: 71862
  CONCEPT_ABSTRACT: 32376
  ENTITY_OBJECT: 25649
  ATTRIBUTE_PROPERTY: 15965
  ACTION_OPERATION: 9557
  GOAL_INTENT: 4586
  ERROR_PROBLEM: 2762
  METHOD_STRATEGY: 2168
  STATE_STATUS: 1554
  TOOL_SYSTEM: 1457


## 4. 训练 Tfidf + LR baseline（轻量）

CPU 即可，2 分钟。

In [14]:
!python src/train_baseline.py

Loaded 359051 samples
Cognitive labels: 12
  ACTION_OPERATION: 9557
  ATTRIBUTE_PROPERTY: 15965
  CONCEPT_ABSTRACT: 32376
  DATA_INFO: 278496
  ENTITY_OBJECT: 25649
  ERROR_PROBLEM: 2762
  GOAL_INTENT: 4586
  LANGUAGE_EXPRESSION: 359051
  METHOD_STRATEGY: 2168
  RELATION_STRUCTURE: 71862
  STATE_STATUS: 1554
  TOOL_SYSTEM: 1457
Feature matrix: (359051, 50000)
/usr/local/lib/python3.12/dist-packages/sklearn/multiclass.py:90: UserWarning: Label 7 is present in all training examples.
  warnings.warn(
Training complete.
Train hamming loss: 0.0985
Train micro-F1:     0.7754
Train macro-F1:     0.4277

Model saved to: /content/outputs/models/source_aware_multilabel_classifier.joblib

Classification report (trainset):
                     precision    recall  f1-score   support

   ACTION_OPERATION       0.29      0.95      0.45      9557
 ATTRIBUTE_PROPERTY       0.36      0.90      0.51     15965
   CONCEPT_ABSTRACT       0.38      0.92      0.54     32376
          DATA_INFO       0.94    

In [ ]:
# baseline 预测示例
examples = ['接口调用失败', 'Agent 调用工具失败', '机器人路径规划']
for t in examples:
    print(f'\n--- {t} ---')
    !python src/predict.py --text "$t"

## 5. 训练 BERT-base-chinese 多标签分类器

Colab T4 GPU，5 万样本 3 epochs，约 10-15 分钟。

选项：
- `--max-samples 50000` 快速实验
- `--min-confidence 0.72` 只用高置信样本（更干净但更少）
- 全量 35 万样本需要 1-2 小时

In [15]:
# 快速实验：5 万样本，3 epochs
!python src/train_transformer.py --model bert-base-chinese --max-samples 50000 --epochs 3 --batch-size 64 --lr 2e-5

Device: cuda
Loaded 50000 samples (min_confidence=0.0)
Label matrix shape: (50000, 12)
Train: 45000  Val: 5000
Loading tokenizer: bert-base-chinese
tokenizer_config.json: 100% 49.0/49.0 [00:00<00:00, 295kB/s]
vocab.txt: 100% 110k/110k [00:00<00:00, 4.19MB/s]
tokenizer.json: 100% 269k/269k [00:00<00:00, 3.42MB/s]
Loading model: bert-base-chinese
config.json: 100% 624/624 [00:00<00:00, 3.51MB/s]
model.safetensors: 100% 412M/412M [00:04<00:00, 85.8MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 6031.05it/s]
[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED

### (可选) 高置信子集训练

用 `--min-confidence 0.72` 过滤掉 34 万条噪声，只训高置信的 ~1.6 万条。标签更干净。

In [ ]:

!python src/train_transformer.py --min-confidence 0.72 --epochs 5 --batch-size 32

Device: cuda
Loaded 15848 samples (min_confidence=0.72)
Label matrix shape: (15848, 12)
Train: 14263  Val: 1585
Loading tokenizer: bert-base-chinese
Loading model: bert-base-chinese
Loading weights: 100% 199/199 [00:00<00:00, 5816.25it/s]
[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  Epoch 1 step 0/446 

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### (可选) 全量训练

35 万样本，3 epochs，T4 GPU 约 1 小时。

In [ ]:
# # 可选: 全量训练
# !python src/train_transformer.py --epochs 3 --batch-size 64

## 6. BERT 预测 + 与 baseline 对比

In [ ]:
examples = ['接口调用失败', 'Agent 调用工具失败', '机器人路径规划', '数据库连接超时', '语义表征定义']
for t in examples:
    print(f'\n===== {t} =====')
    print('--- Tfidf baseline ---')
    !python src/predict.py --text "$t" --json
    print('--- BERT ---')
    !python src/predict_transformer.py --text "$t"

## 7. 下载模型到本地

把训练好的 BERT 模型打包下载回本地。

In [ ]:
import shutil

# 打包 BERT best 模型
if os.path.exists('outputs/models/source_aware_transformer/best'):
    shutil.make_archive('source_aware_transformer_best', 'zip', 'outputs/models/source_aware_transformer/best')
    print('BERT best 模型已打包: source_aware_transformer_best.zip')
    print('点左侧 Files 面板 → 右键下载')
else:
    print('BERT 模型目录不存在，请先跑 step 5')

## 8. 可视化标签分布

(可选) 查看数据集标签分布、confidence 直方图。

In [ ]:
import json
import matplotlib.pyplot as plt
from collections import Counter

samples = []
with open('outputs/universal_zh_classifier.source_aware.samples.jsonl', encoding='utf-8') as f:
    for line in f:
        samples.append(json.loads(line))

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Cognitive labels
cog = Counter()
for s in samples:
    for lb in s['cognitive_labels']:
        cog[lb] += 1
axes[0].barh(list(cog.keys()), list(cog.values()))
axes[0].set_title('Cognitive Labels')
axes[0].invert_yaxis()

# Domain labels
dom = Counter()
for s in samples:
    for lb in s['domain_labels']:
        dom[lb] += 1
axes[1].barh(list(dom.keys()), list(dom.values()))
axes[1].set_title('Domain Labels')
axes[1].invert_yaxis()

# Confidence histogram
confs = [s['confidence'] for s in samples]
axes[2].hist(confs, bins=30)
axes[2].set_title('Confidence Distribution')
axes[2].axvline(0.72, color='red', linestyle='--', label='needs_review threshold')
axes[2].legend()

plt.tight_layout()
plt.show()

## 备注

- **不是金标**：所有标签来自弱监督（关键词规则），`needs_review=true` 占多数，适合冷启动+人工审核。
- **每条可追溯**：`source_repo / source_record_id / raw_fields / license_hint` 全保留。
- **BERT 首次下载** `bert-base-chinese` 约 400MB，Colab 网络快。
- 想提升效果：增强 `src/label_rules.py` 关键词 → `build_dataset.py` → 重新训练。